# Generating structural data from PubChem SMILES mappings
This notebook demonstrates how to export PubChem CIDs from a global inventory, retrieve the corresponding SMILES strings, generate molecular structures and fingerprints, and save the result as a JSONL file for structure-based analysis.

**Prerequisites**
- Download a copy of the [example global inventory](https://github.com/envris/global-inventory-compiler/tree/main/data) from the GIC repository.

## Step 1: Load the inventory and export PubChem CIDs
Create an Inventory instance and write all available PubChem CIDs to a text file. Only substances with associated PubChem CIDs can be carried forward to the structure-generation workflow.

In [ ]:
from gic import Inventory

# Load the precompiled inventory.
inv = Inventory("input/global_inventory.jsonl")

# Export PubChem CIDs without the CID prefix for PubChem ID Exchange.
with open("output/global_inventory_cids.csv", "w", encoding="utf-8") as f:
    for cid in inv.get_identifiers("cid"):
        f.write(cid[3:] + "\n")

## Step 2: Retrieve the CID-to-SMILES mapping from PubChem
Upload the exported CID file to the [PubChem ID Exchange Service](https://pubchem.ncbi.nlm.nih.gov/idexchange/idexchange.cgi), then download the two-column mapping of input CID to output SMILES. In this example, the downloaded file is renamed to *pubchem_smiles.tsv* and placed in the input directory.

## Step 3: Load the mapping and build a DataFrame
Use the generic CSV extractor to read the PubChem CID and SMILES mapping into a DataFrame. CIDs are automatically patched to include the *CID* prefix required by the GIC.

In [ ]:
from gic import GenericCsvExtractor

# Initialise the extractor.
extractor = GenericCsvExtractor()

# Input and output settings.
extractor.input_file = "input/pubchem_smiles.tsv"
extractor.output_file = ""  # No output file
extractor.errors_file = "output/pubchem_smiles_errors.csv"

# Column-to-field mapping.
extractor.columns = {1: "cid", 2: "smiles"}

# File layout settings.
extractor.start_row = 1
extractor.delimiter = "\t"

# Run the extraction.
df = extractor.extract()

## Step 4: Generate molecular structures and fingerprints
Generate RDKit molecule objects from the SMILES strings, then generate RDKit fingerprints from those molecules. This step enables later substructure and similarity-based queries and may take several minutes to complete.

In [ ]:
# Generate RDKit molecule objects and fingerprints.
df.generate("rdk_mol")
df.generate("rdk_rdkfp")

## Step 5: Save the structural data
The RDKit molecule objects and fingerprints are serialised and written to a JSONL file. When the file is reloaded into a DataFrame, the stored values are restored to their appropriate types.

In [ ]:
# Save the structural data for later analysis.
df.to_jsonl("output/pubchem_structures.jsonl")